# *Detección de estrés y esfuerzo físico mediante aprendizaje automático*

In [ ]:
# Imports
import os
import pandas as pd

from datetime import timedelta, timezone

import flirt
import flirt.reader.empatica

import warnings
warnings.filterwarnings('ignore')

from functools import reduce

Usamos la librería Flirt para la lectura y extracción de características en las señales biométricas, ya que el dataset que estamos utilizando forma parte de los datos extraídos por la Wristband Empatica E4 y esta librería está diseñada para este dispositivo

## Definimos rutas

In [ ]:
### Comentar y descomentar conforme se necesite

## Rutas de las carpetas de pacientes

#base_path = 'data/STRESS'
#base_path = 'data/AEROBIC'
base_path = 'data/ANAEROBIC'


## Ruta donde se guardarán las extracciones

#output_base_path = 'data/STRESS_FeatureExtraction'
#output_base_path = 'data/AEROBIC_FeatureExtraction'
output_base_path = 'data/ANAEROBIC_FeatureExtraction'

print("Ruta de pacientes:", base_path)
print("Ruta de extracción:", output_base_path)

Ruta de pacientes: ANAEROBIC
Ruta de extracción: ANAEROBIC_FeatureExtraction


### Modificamos las ventanas de tiempo solo para la carpeta de Stress
# Correr este código SOLO una vez
Esta celda modifica los tags.csv originales de los pacientes en la carpeta de STRESS, por lo que correrlo más de una vez hará que se vuelvan a modificar los tiempos 5 segundos extra cada vez

Esta modificación la hacemos ya que, al tratarse de estrés mental y no esfuerzo físico, las señales biológicas tardan más en presentarse después de la inducción del estrés (pruebas cognitivas), y en la mayoría de las señales que observamos, las alteraciones físicas como aceleración del Ritmo cardiaco y aumento en la Actividad electrodérmica, se presentan un poco después de la prueba.

In [ ]:
# 1. Modificar los archivos tags.csv adelantando el tiempo 5 segundos
# Descomentar para ejecutar una vez, comentar otra vez cuando se haya ejecutado

stress_path = 'data/STRESS'

"""
modificadas = 0
for folder in sorted(os.listdir(base_path)):
    folder_path = os.path.join(base_path, folder)
    if os.path.isdir(folder_path):
        tags_file = os.path.join(folder_path, 'tags.csv')
        if os.path.exists(tags_file):
            try:
                # Leemos el archivo tags que contiene los timestamps (sin encabezado)
                df_tags = pd.read_csv(tags_file, header=None, names=['timestamp'])
                
                # Parseamos el texto como Fechas/Tiempos
                df_tags['timestamp'] = pd.to_datetime(df_tags['timestamp'], format='%Y-%m-%d %H:%M:%S', errors='coerce')
                
                # Dropeamos (eliminamos) filas vacías que puedan existir al final del csv
                df_tags = df_tags.dropna()
                
                # Adelantar 5 segundos (sumar)
                df_tags['timestamp'] = df_tags['timestamp'] + timedelta(seconds=5)
                
                # Convertimos de nuevo al formato original de texto y sobreescribimos el tags.csv
                df_tags['timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S').to_csv(tags_file, index=False, header=False)
                
                modificadas += 1
            except Exception as e:
                print(f'Error en el folder {folder}: {e}')
                
print(f'Se han adelantado 5 segundos los tags.csv en {modificadas} carpetas.')
"""

# Configuración y corrección de formatos de tiempo
Definimos una función auxiliar para leer las marcas de tiempo (tags.csv)

La función fix_empatica_file() nos sirve para modificar el formato de tiempo de datetime a formato UNIX, que es el que necesita la librería Flirt para poder analizar las señales correctamente

Omitimos pacientes de los datasets originales por contener datos corruptos o señales no válidas

### Causas de omisión de pacientes:

Sesión de estrés: S02 tiene datos duplicados; f07 no retiró la cubierta protectora de la pulsera, por lo que no todas las señales son válidas; los datos de f14 están divididos en dos partes.

Sesión aeróbica: S03 y S07 no pudieron completar el procedimiento; los datos de S11 están divididos en dos partes; S12 no realizó esta prueba.

Sesión anaeróbica: S06 no pudo completar el procedimiento; los datos de S16 están divididos en dos partes.

In [ ]:
def load_tags(tags_file):
    """Carga los tags de tiempo y los localiza en UTC para coincidir con Flirt"""
    tags = pd.read_csv(tags_file, header=None, names=['timestamp'])
    tags['timestamp'] = pd.to_datetime(tags['timestamp'], format='%Y-%m-%d %H:%M:%S', errors='coerce').dt.tz_localize('UTC')
    tags = tags.dropna()
    return tags['timestamp'].tolist()

def fix_empatica_file(filepath, filename):
    with open(filepath, 'r') as f:
        lineas = f.readlines()
        
    fecha_str = lineas[0].split(',')[0].strip()
    
    try:
        timestamp_unix = pd.to_datetime(fecha_str, utc=True).timestamp()
    except Exception as e:
        return False
        
    if filename == 'IBI.csv':
        lineas[0] = f"{timestamp_unix}, IBI\n"
    elif filename == 'ACC.csv':
        lineas[0] = f"{timestamp_unix},{timestamp_unix},{timestamp_unix}\n"
    else:
        lineas[0] = f"{timestamp_unix}\n"
            
    with open(filepath, 'w') as f:
        f.writelines(lineas)
    return True

# Lista de archivos a descartar
archivos_ignorar = ['tags.csv', 'BVP.csv']

#Lista de pacientes a ignorar dependiendo de la sesión
pacientes_ignorar_stress = ['S02', 'f07', 'f14_a', 'f14_b']
pacientes_ignorar_aerobic = ['S03', 'S07', 'S11_a', 'S11_b']
pacientes_ignorar_anaerobic = ['S06', 'S16_a', 'S16_b']

# Asignamos variable "pacientes_ignorar" dependiendo de la base_path asignada al principio del notebook
if base_path == "data/STRESS":
    pacientes_ignorar = pacientes_ignorar_stress
elif base_path == "data/AEROBIC":
    pacientes_ignorar = pacientes_ignorar_aerobic
elif base_path == "data/ANAEROBIC":
    pacientes_ignorar = pacientes_ignorar_anaerobic
else:
    print("Carpeta invalida")

print("--- INICIANDO CORRECCIÓN DE FORMATO DE TIEMPO ---")

for folder in sorted(os.listdir(base_path)):
    if folder in pacientes_ignorar:
        print(f"  -> Excluyendo paciente {folder}")
        continue
    folder_path = os.path.join(base_path, folder)
    if os.path.isdir(folder_path):
        archivos = [f for f in os.listdir(folder_path) if f.endswith('.csv') and f not in archivos_ignorar and not f.startswith('flirt_')]
        for archivo in archivos:
            filepath = os.path.join(folder_path, archivo)
            fix_empatica_file(filepath, archivo)
            
print("Formato corregido en todos los archivos pertinentes.")

--- INICIANDO CORRECCIÓN DE FORMATO DE TIEMPO ---
  -> Excluyendo paciente S06
  -> Excluyendo paciente S16_a
  -> Excluyendo paciente S16_b
Formato corregido en todos los archivos pertinentes.


# Carga de datos usando los Flirt readers 

In [11]:
datos_flirt = {} 

print("--- CARGANDO DATOS CON FLIRT READERS ---")
for folder in sorted(os.listdir(base_path)):
    if folder in pacientes_ignorar:
        continue

    folder_path = os.path.join(base_path, folder)
    if os.path.isdir(folder_path):
        datos_flirt[folder] = {}
        archivos = [f for f in os.listdir(folder_path) if f.endswith('.csv') and f not in archivos_ignorar and not f.startswith('flirt_')]
        
        for archivo in archivos:
            filepath = os.path.join(folder_path, archivo)
            try:
                if archivo == 'IBI.csv':
                    df = flirt.reader.empatica.read_ibi_file_into_df(filepath)
                elif archivo == 'EDA.csv':
                    df = flirt.reader.empatica.read_eda_file_into_df(filepath)
                elif archivo == 'ACC.csv':
                    df = flirt.reader.empatica.read_acc_file_into_df(filepath)
                elif archivo == 'TEMP.csv':
                    df = flirt.reader.empatica.read_temp_file_into_df(filepath)
                elif archivo == 'HR.csv':
                    df = flirt.reader.empatica.read_hr_file_into_df(filepath)
                else:
                    continue
                    
                datos_flirt[folder][archivo] = df
            except Exception as e:
                print(f"Error cargando {archivo} en {folder}: {e}")
                
print("Datos cargados exitosamente")

--- CARGANDO DATOS CON FLIRT READERS ---
Datos cargados exitosamente


# Extracción de características

In [ ]:
print("--- INICIANDO EXTRACCIÓN DE CARACTERÍSTICAS ---")

for folder, archivos_dict in datos_flirt.items():
    folder_path = os.path.join(base_path, folder)
    output_folder_path = os.path.join(output_base_path, folder)
    os.makedirs(output_folder_path, exist_ok=True)
    
    tags_file = os.path.join(folder_path, 'tags.csv')
    if not os.path.exists(tags_file):
        print(f"  - Sin tags.csv en {folder}, omitiendo.")
        continue
        
    tags_list = load_tags(tags_file)
    
    for archivo, df in archivos_dict.items():
        feature_file = os.path.join(output_folder_path, f'flirt_{archivo}')
        
        # Comprobar si el archivo ya fue extraído previamente
        if os.path.exists(feature_file):
            print(f"  -> Omitiendo {archivo} en {folder} (El archivo ya existe)")
            continue    
        
        start_time = df.index[0]
        limites = tags_list.copy()
        limites.append(df.index[-1])
        
        segment_features_list = []
        
        for idx, tag_time in enumerate(limites):
            if start_time >= df.index[-1] or start_time >= tag_time:
                start_time = tag_time
                continue
                
            segment_df = df.loc[start_time:tag_time]
            duration = int((tag_time - start_time).total_seconds())
            
            if len(segment_df) > 0 and duration > 0:
                try:
                    if archivo == 'IBI.csv':
                        feat = flirt.get_hrv_features(
                            segment_df['ibi'], 
                            window_length=duration, 
                            window_step_size=duration,
                            domains=['td', 'fd', 'nl', 'stat'], 
                            threshold=0.5, 
                            clean_data=True
                        )
                    elif archivo == 'EDA.csv':
                        feat = flirt.get_eda_features(segment_df['eda'], window_length=duration, window_step_size=duration)
                    elif archivo == 'ACC.csv':
                        feat = flirt.get_acc_features(segment_df, window_length=duration, window_step_size=duration)
                    else:
                        feat = flirt.get_stat_features(segment_df, window_length=duration, window_step_size=duration)
                    
                    if not feat.empty:
                        feat['Segmento_ID'] = idx 
                        segment_features_list.append(feat)
                        
                except Exception as e:
                    pass # Omitir silenciosamente si no hay datos suficientes
            
            start_time = tag_time
            
        # Guardado de archivos
        if segment_features_list:
            all_features = pd.concat(segment_features_list)
            feature_cols = [col for col in all_features.columns if col != 'Segmento_ID']
            all_features.dropna(subset=feature_cols, how='all', inplace=True)
            
            if not all_features.empty:
                feature_file = os.path.join(output_folder_path, f'flirt_{archivo}')
                all_features.to_csv(feature_file)
                print(f"  ✓ Extraído y guardado: {archivo} ({folder})")
                
print("Extracción completada")

# Mapeo de etiquetas

In [5]:
"""
0: Reposo / Descanso
1: Estrés
2: Ejercicio anaeróbico
3: Ejercicio aeróbico (Descartado)
"""


# Mapeo para pacientes S (Primera versión del protocolo)
mapeo_etiquetas_Stress_S = {
    0: 0, # Inicio a Tag 1 (Baseline)
    1: 0, # Tag 1 a Tag 2 (Baseline)
    2: 0, # Tag 2 a Tag 3 (Baseline/Espera)
    3: 1, # Tag 3 a Tag 4 [ESTRÉS: Stroop]
    4: 0, # Tag 4 a Tag 5 (Espera)
    5: 1, # Tag 5 a Tag 6 [ESTRÉS: TMCT]
    6: 0, # Tag 6 a Tag 7 (First Rest)
    7: 1, # Tag 7 a Tag 8 [ESTRÉS: Real Opinion]
    8: 0, # Tag 8 a Tag 9 (Espera)
    9: 1, # Tag 9 a Tag 10 [ESTRÉS: Opposite Opinion]
    10: 0, # Tag 10 a Tag 11 (Espera)
    11: 1, # Tag 11 a Tag 12 [ESTRÉS: Subtract Test]
    12: 0, # Tag 12 al final de la señal (Recuperación final)
    13: 0
}

# Mapeo para pacientes f (Segunda versión del protocolo)
mapeo_etiquetas_Stress_F = {
    0: 0, # Inicio a Tag 1 (Baseline)
    1: 0, # Tag 1 a Tag 2 (Baseline)
    2: 1, # Tag 2 a Tag 3 [ESTRÉS: TMCT]
    3: 0, # Tag 3 a Tag 4 (First Rest)
    4: 1, # Tag 4 a Tag 5 [ESTRÉS: Real Opinion]
    5: 0, # Tag 5 a Tag 6 (Espera)
    6: 1, # Tag 6 a Tag 7 [ESTRÉS: Opposite Opinion]
    7: 0, # Tag 7 a Tag 8 (Second Rest)
    8: 1, # Tag 8 a Tag 9 [ESTRÉS: Subtract Test]
    9: 0, # Tag 9 al final de la señal (Recuperación final)
}

mapeo_etiquetas_Anaerobic_S = {
    0: 0,
    1: 2, # Sprint 1
    2: 0,
    3: 2, # Sprint 2
    4: 0,
    5: 2, # Sprint 3
    6: 0,
    7: 0,
}

mapeo_etiquetas_Anaerobic_F = {
    0: 0,
    1: 0,
    2: 2, # Sprint 1
    3: 0,
    4: 2, # Sprint 2
    5: 0,
    6: 2, # Sprint 3
    7: 0,
    8: 2, # Sprint 4
    9: 0,
    10: 0,
    11: 0
}

"""
mapeo_etiquetas_Aerobic_S = {
    0: '',
    1: '',
    2: '',
    3: '' 
}

mapeo_etiquetas_Aerobic_F = {
    0: '',
    1: '',
    2: '',
    3: '' 
}
"""

"\nmapeo_etiquetas_Aerobic_S = {\n    0: '',\n    1: '',\n    2: '',\n    3: '' \n}\n\nmapeo_etiquetas_Aerobic_F = {\n    0: '',\n    1: '',\n    2: '',\n    3: '' \n}\n"

# Asignación de etiquetas

In [ ]:
output_base_path = 'data/STRESS_FeatureExtraction'
#output_base_path = 'data/ANAEROBIC_FeatureExtraction'

# Iterar sobre los datos extraídos 
for folder in sorted(os.listdir(output_base_path)):
    output_folder_path = os.path.join(output_base_path, folder)
    
    if os.path.isdir(output_folder_path):
        # Determinar qué diccionario usar basándose en la primera letra de la carpeta
        if folder.startswith('S'):
            diccionario_actual = mapeo_etiquetas_Stress_S           # Estrés
            #diccionario_actual = mapeo_etiquetas_Anaerobic_S       # Ejercicio Anaerobico
            version = "Versión 1 (S)"
        
        elif folder.startswith('f'):
            diccionario_actual = mapeo_etiquetas_Stress_F           # Estrés
            #diccionario_actual = mapeo_etiquetas_Anaerobic_F       # Ejercicio Anaerobico
            version = "Versión 2 (f)"
        
        else:
            print(f"  - Omitiendo carpeta desconocida: {folder}")
            continue
            
        # Buscar los archivos generados por Flirt
        archivos_flirt = [f for f in os.listdir(output_folder_path) if f.startswith('flirt_') and f.endswith('.csv')]
        
        for archivo in archivos_flirt:
            filepath = os.path.join(output_folder_path, archivo)
            
            try:
                # Cargar el CSV (index_col=0 asegura que el DatetimeIndex se mantenga como índice)
                df_feat = pd.read_csv(filepath, index_col=0)
                
                # Verificar que exista la columna Segmento_ID antes de mapear
                if 'Segmento_ID' in df_feat.columns:
                    # Mapear los IDs a sus etiquetas 
                    # El .get(x, 0) asegura que si por alguna razón hay un ID extra (ej. al final de la señal), se clasifique como 0 (Reposo)
                    df_feat['Estado'] = df_feat['Segmento_ID'].map(lambda x: diccionario_actual.get(x, 0))

                    # Eliminar la columna temporal Segmento_ID para dejar el dataset limpio
                    df_feat.drop(columns=['Segmento_ID'], inplace=True)
                    
                    # Sobrescribir el archivo CSV con las nuevas columnas
                    df_feat.to_csv(filepath)
                    
            except Exception as e:
                print(f"Error etiquetando {archivo} en {folder}: {e}")
                
        print(f"  ✓ Etiquetas asignadas en {folder} usando mapeo de {version}")
        
print("\nEtiquetado completo con éxito en todos los datasets")

  ✓ Etiquetas asignadas en S01 usando mapeo de Versión 1 (S)
  ✓ Etiquetas asignadas en S02 usando mapeo de Versión 1 (S)
  ✓ Etiquetas asignadas en S03 usando mapeo de Versión 1 (S)
  ✓ Etiquetas asignadas en S04 usando mapeo de Versión 1 (S)
  ✓ Etiquetas asignadas en S05 usando mapeo de Versión 1 (S)
  ✓ Etiquetas asignadas en S07 usando mapeo de Versión 1 (S)
  ✓ Etiquetas asignadas en S08 usando mapeo de Versión 1 (S)
  ✓ Etiquetas asignadas en S09 usando mapeo de Versión 1 (S)
  ✓ Etiquetas asignadas en S10 usando mapeo de Versión 1 (S)
  ✓ Etiquetas asignadas en S11 usando mapeo de Versión 1 (S)
  ✓ Etiquetas asignadas en S12 usando mapeo de Versión 1 (S)
  ✓ Etiquetas asignadas en S13 usando mapeo de Versión 1 (S)
  ✓ Etiquetas asignadas en S14 usando mapeo de Versión 1 (S)
  ✓ Etiquetas asignadas en S15 usando mapeo de Versión 1 (S)
  ✓ Etiquetas asignadas en S17 usando mapeo de Versión 1 (S)
  ✓ Etiquetas asignadas en S18 usando mapeo de Versión 1 (S)
  ✓ Etiquetas asignadas 

# Unión de características en un dataset masivo

In [ ]:
# Definimos las rutas y prefijos con los que guardaran los IDs
estudios = [
    {"ruta": "data/STRESS_FeatureExtraction", "prefijo": "S_"},
    {"ruta": "data/ANAEROBIC_FeatureExtraction", "prefijo": "A_"}
]

all_patients_data = []
tolerancia_tiempo = pd.Timedelta("3s")

# Iterar sobre la lista de estudios
for estudio in estudios:
    base_path = estudio["ruta"]
    prefijo = estudio["prefijo"]

    if not os.path.exists(base_path):
        print(f"Advertencia: No se encontró la carpeta '{base_path}'. Omitiendo...")
        continue

    # Iterar sobre las carpetas de pacientes dentro del estudio actual
    for px_id in os.listdir(base_path):
        patient_folder = os.path.join(base_path, px_id)
        
        if os.path.isdir(patient_folder):
            
            # Filtrar archivos (excluyendo IBI)
            csv_files = [
                os.path.join(patient_folder, f) 
                for f in os.listdir(patient_folder) 
                if f.startswith("flirt_") and f.endswith(".csv") and f != "flirt_IBI.csv"
            ]
            
            if not csv_files:
                continue
                
            patient_dfs = []
            for i, file_path in enumerate(csv_files):
                df = pd.read_csv(file_path)
                
                # Convertir a datetime y ordenar
                df['datetime'] = pd.to_datetime(df['datetime'])
                df = df.sort_values('datetime').reset_index(drop=True)
                
                # Manejo de la variable objetivo "Estado"
                if i > 0 and 'Estado' in df.columns:
                    df = df.drop(columns=['Estado'])
                
                patient_dfs.append(df)
                
            if not patient_dfs:
                continue

            # Fusionar con tolerancia de tiempo
            merged_patient_df = reduce(
                lambda left, right: pd.merge_asof(
                    left, 
                    right, 
                    on='datetime', 
                    direction='nearest', 
                    tolerance=tolerancia_tiempo
                ), 
                patient_dfs
            )
            
            merged_patient_df = merged_patient_df.dropna()
            
            # Añadir la columna PxID concatenando el PREFIJO del estudio y el ID original
            merged_patient_df.insert(0, 'PxID', f"{prefijo}{px_id}")
            
            all_patients_data.append(merged_patient_df)

# Unir todos los pacientes de TODOS los estudios
if all_patients_data:
    final_massive_df_tolerant = pd.concat(all_patients_data, ignore_index=True)
    
    # Ordenar por el nuevo PxID (que incluye el prefijo) y luego por tiempo
    final_massive_df_tolerant = final_massive_df_tolerant.sort_values(by=['PxID', 'datetime']).reset_index(drop=True)
    
    # Mover la columna "Estado" al final del dataframe masivo
    if 'Estado' in final_massive_df_tolerant.columns:
        columnas_sin_estado = [col for col in final_massive_df_tolerant.columns if col != 'Estado']
        final_massive_df_tolerant = final_massive_df_tolerant[columnas_sin_estado + ['Estado']]
    
    print("\n--- Dataset listo ---")
    print(f"Total de registros conservados: {len(final_massive_df_tolerant)}")
    print(f"Total de pacientes únicos procesados: {final_massive_df_tolerant['PxID'].nunique()}")
    
    final_massive_df_tolerant.to_csv("SignalFeatures_Dataset.csv", index=False)
else:
    print("No se encontraron datos para procesar. Verifica las rutas de tus carpetas.")

final_massive_df_tolerant.head()


--- Dataset listo ---
Total de registros conservados: 662
Total de pacientes únicos procesados: 62


,PxID,datetime,acc_x_mean,acc_x_std,acc_x_min,acc_x_max,acc_x_ptp,acc_x_sum,acc_x_energy,acc_x_skewness,...,temp_n_below_mean,temp_n_sign_changes,temp_iqr,temp_iqr_5_95,temp_pct_5,temp_pct_95,temp_entropy,temp_perm_entropy,temp_svd_entropy,Estado
0,A_S01,2013-02-20 18:45:54+00:00,-54.448550,1.984517,-63.0,-48.0,15.0,-315366.0,17194032.0,-0.221483,...,512.0,0.0,0.04,0.14,31.83,31.97,6.584790,0.737728,0.003075,0
1,A_S01,2013-02-20 18:46:36+00:00,-55.099702,3.778920,-75.0,-37.0,38.0,-74054.0,4099546.0,-0.145297,...,64.0,0.0,0.04,0.06,31.79,31.85,5.123964,0.892121,0.003681,2
2,A_S01,2013-02-20 18:50:47+00:00,-55.522286,2.799447,-77.0,-41.0,36.0,-445955.0,24823387.0,-0.076732,...,320.0,0.0,0.04,0.10,31.79,31.89,6.911747,0.774108,0.003362,0
3,A_S01,2013-02-20 18:51:20+00:00,-57.663826,4.065708,-72.0,-40.0,32.0,-60893.0,3528779.0,0.186199,...,68.0,0.0,0.02,0.04,31.77,31.81,4.882802,0.740147,0.003050,2
4,A_S01,2013-02-20 18:55:24+00:00,-48.999103,4.633148,-128.0,55.0,183.0,-382585.0,18913929.0,0.038950,...,576.0,0.0,0.90,1.10,31.63,32.73,6.883370,0.716551,0.003655,0


# Prueba unitaria para archivo IBI 
### Experimental: No forma parte del dataset final

In [ ]:
# 1. Leer todas las líneas del archivo original
with open('IBI.csv', 'r') as f:
    lineas = f.readlines()

# 2. Extraer la fecha del primer string y convertirla a timestamp UNIX
# Se extrae la primera parte antes de la coma: '2013-06-12 18:28:42'
fecha_str = lineas[0].split(',')[0] 

# Convertimos a datetime y luego a timestamp UNIX (forzando UTC como lo requiere Flirt)
timestamp_unix = pd.to_datetime(fecha_str).replace(tzinfo=timezone.utc).timestamp()

# 3. Reemplazar únicamente la primera línea con el formato esperado por el reader
lineas[0] = f"{timestamp_unix}, IBI\n"

# 4. Guardar el archivo corregido, listo para usar
with open('IBI_corregido.csv', 'w') as f:
    f.writelines(lineas)

In [ ]:
import flirt
import flirt.reader.empatica

# 1. Leer el archivo IBI corregido usando el reader nativo de Flirt
# Esto leerá el timestamp UNIX inicial, saltará esa fila, y automáticamente escalará los IBI a milisegundos creando el DatetimeIndex correcto.
ibi_df = flirt.reader.empatica.read_ibi_file_into_df('IBI_corregido.csv')

# 2. Extraer las características HRV
# Pasamos específicamente la serie 'ibi' del DataFrame a la función.
hrv_features = flirt.get_hrv_features(
    ibi_df['ibi'], 
    window_length=180, # Tamaño de la ventana en segundos
    window_step_size=1, # Tamaño del paso para la ventana deslizante
    domains=['td', 'fd', 'nl', 'stat'], # Dominios: tiempo, frecuencia, no lineal y estadístico
    threshold=0.5, # Porción relativa de IBIs necesarios en una ventana
    clean_data=True # Mantenemos la limpieza activada, ahora funcionará bien porque los datos están en milisegundos
)

# 3. Visualizar los resultados de la extracción
print(hrv_features.head())

In [ ]:
# 4. guardar características
hrv_features.to_csv("extracción_IBI.csv")